In [6]:
%pip install transformers datasets accelerate scikit-learn pandas matplotlib torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

SEED = 42
MODEL_NAME = "FacebookAI/roberta-base"
LABELS = ["left", "center", "right"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cpu
CUDA available: False


In [8]:
DATA_PATH = Path("../data/processed/newsbias_cleaned.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClass distribution:")
print(df["bias_rating"].value_counts())

print("\nMissing model text:", df["model_text"].isna().sum())
print("Missing labels:", df["bias_rating"].isna().sum())
print("Empty model inputs:", df["model_text"].fillna("").str.strip().eq("").sum())

assert df["model_text"].notna().all()
assert df["bias_rating"].notna().all()
assert set(df["bias_rating"].unique()) == set(LABELS)

print("\nDataset validation passed.")

Dataset shape: (21748, 9)

Columns:
['Unnamed: 0', 'title', 'tags', 'heading', 'source', 'text', 'bias_rating', 'model_text', 'normalized_model_text']

Class distribution:
bias_rating
left      10271
right      7225
center     4252
Name: count, dtype: int64

Missing model text: 0
Missing labels: 0
Empty model inputs: 0

Dataset validation passed.


In [9]:
DATA_PATH = Path("../data/processed/newsbias_cleaned.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClass distribution:")
print(df["bias_rating"].value_counts())

print("\nMissing model text:", df["model_text"].isna().sum())
print("Missing labels:", df["bias_rating"].isna().sum())
print("Empty model inputs:", df["model_text"].fillna("").str.strip().eq("").sum())

assert df["model_text"].notna().all()
assert df["bias_rating"].notna().all()
assert set(df["bias_rating"].unique()) == set(LABELS)

print("\nDataset validation passed.")

Dataset shape: (21748, 9)

Columns:
['Unnamed: 0', 'title', 'tags', 'heading', 'source', 'text', 'bias_rating', 'model_text', 'normalized_model_text']

Class distribution:
bias_rating
left      10271
right      7225
center     4252
Name: count, dtype: int64

Missing model text: 0
Missing labels: 0
Empty model inputs: 0

Dataset validation passed.


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)

Tokenizer loaded: FacebookAI/roberta-base


In [11]:
import os
import multiprocessing

print("CPU cores:", multiprocessing.cpu_count())
print("Available RAM (GB):", round(
    __import__("psutil").virtual_memory().available / (1024**3), 2
))

CPU cores: 8
Available RAM (GB): 2.33
